# 02 — Training Benchmark: Delta vs Lance

**Purpose:** Read each format back through Ray Data and feed it to Ray Train, measuring where the storage format separates the two pipelines. Covers stages **4 (inline preprocess), 5 (train), 6 (compare)** of [`README.md`](README.md).

**Prerequisite:** For the target `size` tier, run both paved-path notebooks first — `01a_delta_native.ipynb` (produces `synthetic_delta_{size}`) and `01b_lance_native.ipynb` (produces `synthetic_lance_{size}`).

| Branch | Read path |
|--------|-----------|
| **Lance** | `read_lance` → image bytes inline, no extra hop |
| **Delta** | `read_databricks_tables` (metadata + `image_path`) → per-image Volumes GET to fetch JPEG bytes |
| **Delta inline** *(optional)* | `read_databricks_tables` (`image` binary column) → bytes ship through the SQL Warehouse; the anti-pattern, off by default |

The Delta per-image GET hop is the mechanism the parent blueprint's failure-mode #1 describes. At batch 64 that's 64 concurrent file reads per step — the thing that starves the GPU. Lance reads bytes directly.

| Section | What it measures |
|---------|------------------|
| **1 — Training throughput** | Dummy-model (data-loading ceiling) and real ResNet-50 runs; samples/sec, time-to-first-batch, batch-latency p50/p95/p99, inter-batch wait, effective GPU util |
| **2 — Compare** | Side-by-side, logged to MLflow |

Set the `include_inline_delta` widget to `true` to add a third branch that reads the **inline** Delta table from `01a` (JPEG bytes in a `binary` column). It needs `01a` run with `write_inline_delta=true` for the same `size`. This measures the read side of the anti-pattern the parent README describes — expected to be slower than *both* other branches at scale from Parquet row-group collapse. Default off.

**Why no separate data-loading pass?** We only care how the storage format feeds training. The **dummy** run streams and decodes every batch but skips the GPU step, so it *is* the data-loading ceiling — no need for a redundant non-GPU full pass over each format.

**Shuffle: streaming, not global.** A global `random_shuffle()` is a blocking all-to-all barrier — the GPUs idle until the whole dataset is read and reshuffled, and it reads sequentially (a full scan), which is exactly the pattern where the two formats look *most alike*. Instead we combine `randomize_block_order()` (cheap per-epoch read-order entropy that actually exercises Lance's O(1) random access vs Delta row-group scans) with a streaming `local_shuffle_buffer` in `iter_torch_batches` (per-row entropy from a sliding buffer). Pipelined, non-blocking, and it sharpens rather than hides the format difference.

---

### GPU sizing

Transfer-learning **ResNet-50** on an **A10 (24GB)**: ~600–900 img/s/GPU at 224px with AMP — light compute, so a single GPU is easy to *starve*. That is what surfaces a data-loading bottleneck. Start at **4 × A10** (`num_workers=4`) → ~2.4–3.6k img/s aggregate demand, the range where the Delta GET hop and shuffled random-access reads diverge from Lance. Bump `num_gpu_workers` to 8 for the 1m/10m tiers.

In [0]:
%pip install -qU "ray[default,data,train]==2.56.1" pylance==9.0.0
dbutils.library.restartPython()

In [0]:
# ── Widgets ───────────────────────────────────────────────────────────────
dbutils.widgets.dropdown("size", "10k", ["10k", "100k", "1m", "10m"], "Dataset size")
dbutils.widgets.text("catalog", "main", "UC catalog")
dbutils.widgets.text("schema", "ml_benchmark", "UC schema")
dbutils.widgets.text("volume", "lance_benchmark", "UC volume")
dbutils.widgets.text("warehouse_id", "", "SQL Warehouse ID (blank = provision)")
dbutils.widgets.text("num_gpu_workers", "2", "GPU workers (A10)")
dbutils.widgets.text("num_epochs", "3", "Epochs (steady-state)")
dbutils.widgets.dropdown("include_inline_delta", "false", ["false", "true"], "Include inline-Delta branch (anti-pattern)")
dbutils.widgets.text("mlflow_experiment", "", "MLflow experiment (blank = default)")

size            = dbutils.widgets.get("size")
catalog         = dbutils.widgets.get("catalog")
schema          = dbutils.widgets.get("schema")
volume          = dbutils.widgets.get("volume")
NUM_GPU_WORKERS = int(dbutils.widgets.get("num_gpu_workers"))
NUM_EPOCHS      = int(dbutils.widgets.get("num_epochs"))
INCLUDE_INLINE_DELTA = dbutils.widgets.get("include_inline_delta") == "true"

# MUST match 01a_delta_native + 01b_lance_native.
CATEGORIES = ["cat", "dog", "car", "tree", "house", "flower", "boat", "bird"]
IMG_SIZE   = 224

base_vol     = f"/Volumes/{catalog}/{schema}/{volume}"
lance_path   = f"{base_vol}/synthetic_lance_{size}"
delta_table  = f"{catalog}.{schema}.synthetic_delta_{size}"
inline_delta_table = f"{catalog}.{schema}.synthetic_delta_inline_{size}"  # optional, from 01a
ray_tmp_path = f"/Volumes/{catalog}/{schema}/ray_tmp"

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
username      = notebook_path.split("/")[2]
mlflow_exp    = dbutils.widgets.get("mlflow_experiment") or f"/Users/{username}/delta-vs-lance-benchmark"

print(f"Size tier   : {size}")
print(f"GPU workers : {NUM_GPU_WORKERS} x A10")
print(f"Lance       : {lance_path}")
print(f"Delta table : {delta_table}")
FORMATS = ["delta", "lance"] + (["delta_inline"] if INCLUDE_INLINE_DELTA else [])
print(f"Formats     : {FORMATS}")
print(f"MLflow exp  : {mlflow_exp}")

## Data-loading / GPU-utilization knobs

One block for every parameter that governs how fast data reaches the GPU. These are **held constant across the Delta and Lance runs** — that's the whole point of the benchmark, so they're tuning constants here, not per-run widgets. Tune them once for your cluster, then leave them fixed while you sweep `size` / `num_gpu_workers`.

| Knob | Effect | Cost of raising |
|------|--------|-----------------|
| `PREFETCH_BATCHES` | Overlaps host→device copy + next-batch prep with the current GPU step — the single biggest anti-starvation lever. | Host RAM for prefetched batches. |
| `LOCAL_SHUFFLE_BUFFER` | Sliding-window per-row shuffle entropy in `iter_torch_batches`. | Host RAM (~`buffer × 224²×3×4B` decoded). |
| `BATCH_SIZE` | Decode vectorization + GPU step size. | GPU memory. |

In [0]:
# ── Data-loading / GPU-utilization knobs (held constant across Delta & Lance) ──
BATCH_SIZE           = 64     # decode vectorization + GPU step size
PREFETCH_BATCHES     = 4      # batches prefetched ahead of the GPU in iter_torch_batches
LOCAL_SHUFFLE_BUFFER = 4096   # streaming per-row shuffle buffer (rows)

In [0]:
import os

os.environ["DATABRICKS_HOST"]  = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
os.environ["DATABRICKS_TOKEN"] = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

## SQL Warehouse — provision or reuse

`ray.data.read_databricks_tables` (the Delta branch) routes through a running SQL Warehouse. Same provision-or-reuse helper as `01a`.

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import State

w = WorkspaceClient()
WAREHOUSE_NAME = "ray-benchmark-warehouse"


def get_or_create_warehouse(warehouse_id="", name=WAREHOUSE_NAME,
                            cluster_size="Small", auto_stop_mins=10):
    if warehouse_id:
        return warehouse_id
    for wh in w.warehouses.list():
        if wh.name == name:
            if wh.state in (State.STOPPED, State.STOPPING):
                w.warehouses.start(wh.id).result()
            elif wh.state == State.STARTING:
                w.warehouses.get_and_wait(wh.id)
            print(f"Reusing warehouse '{name}' ({wh.id})")
            return wh.id
    created = w.warehouses.create(
        name=name, cluster_size=cluster_size, auto_stop_mins=auto_stop_mins,
        enable_serverless_compute=True, min_num_clusters=1, max_num_clusters=1,
    ).result()
    print(f"Created warehouse '{name}' ({created.id})")
    return created.id


warehouse_id = get_or_create_warehouse(dbutils.widgets.get("warehouse_id"))

In [0]:
# GPU Ray cluster — pure Ray, no Spark dependency.
# Set spark.task.resource.gpu.amount = "0" in the cluster Spark config so Ray
# (not Spark) controls GPU allocation. One A10G per g5.2xlarge worker node.


import ray
from ray.util.spark import setup_ray_cluster, shutdown_ray_cluster

try:
    shutdown_ray_cluster()
except Exception:
    pass

setup_ray_cluster(
    min_worker_nodes=NUM_GPU_WORKERS,
    max_worker_nodes=NUM_GPU_WORKERS,
    num_gpus_worker_node=1,
    num_cpus_worker_node=16,    # g5.4xlarge = 16 vCPU per node
    collect_log_to_path=ray_tmp_path,
)
ctx = ray.init(address="auto", ignore_reinit_error=True)

total_gpus = ray.cluster_resources().get("GPU", 0)
print(f"Total GPUs  : {total_gpus:.0f}")
print(f"Ray Dashboard: {ctx.dashboard_url}")
assert total_gpus >= NUM_GPU_WORKERS, "GPUs missing — check spark.task.resource.gpu.amount = '0'"

## Shared preprocessing + readers

`decode_resize` fuses JPEG-decode → resize → normalize into the read path (CPU actors, parallel to GPU training). The two readers differ exactly where the formats differ:

- **Lance** — `read_lance(columns=["image","category"])`, bytes inline.
- **Delta** — `read_databricks_tables(query=...)` returns `image_path` + `category`, then `read_image_from_path` issues the per-image Volumes GET. This is the extra hop the benchmark isolates.

Column projection (`image`/`image_path` + `category` only — skipping caption/embedding/metadata) is where Lance's blob isolation shows up: metadata columns are never touched.

In [0]:
import numpy as np

CAT_TO_IDX = {c: i for i, c in enumerate(CATEGORIES)}
io_batch_size = 64

def decode_resize(batch, img_size, cat_to_idx, augment=True):
    """Fused decode → resize → augment → normalize. Expects batch['image'] as JPEG bytes.

    Training augmentations (when augment=True):
      - Random horizontal flip (p=0.5)
      - Random rotation ±15°
      - Color jitter (brightness, contrast, saturation)
      - Gaussian noise (σ=0.02)
    These are standard transforms for improving generalisation on image classifiers.
    """
    import io
    from PIL import Image, ImageFilter
    import random
    from torchvision import transforms as T

    # Build the augmentation pipeline (torchvision transforms on PIL images).
    # Keep it deterministic per-worker by not seeding — each batch gets different augments.
    if augment:
        aug = T.Compose([
            T.RandomHorizontalFlip(p=0.5),
            T.RandomRotation(degrees=15),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
        ])
    else:
        aug = None

    imgs, labels = [], []
    for jpeg, cat in zip(batch["image"], batch["category"]):
        img = Image.open(io.BytesIO(bytes(jpeg))).convert("RGB").resize((img_size, img_size))

        # Apply augmentations
        if aug is not None:
            img = aug(img)

        arr = np.asarray(img, dtype=np.float32) / 255.0

        # Gaussian noise (applied on the normalised array for controlled σ)
        if augment:
            arr = arr + np.random.normal(0, 0.02, arr.shape).astype(np.float32)
            arr = np.clip(arr, 0.0, 1.0)

        arr = arr.transpose(2, 0, 1)  # HWC → CHW
        imgs.append(arr)
        labels.append(cat_to_idx[cat if isinstance(cat, str) else cat.decode()])
    return {"image": np.asarray(imgs, dtype=np.float32),
            "label": np.asarray(labels, dtype=np.int64)}


def read_image_from_path(batch):
    """Delta branch: fetch JPEG bytes from each image_path (per-image Volumes GET)."""
    out = []
    for p in batch["image_path"]:
        with open(p, "rb") as f:
            out.append(f.read())
    return {"image": np.asarray(out, dtype=object), "category": batch["category"]}


def read_format(fmt, warehouse_id, catalog, schema, delta_table, lance_path,
                inline_delta_table=None):
    """Branch-specific reader producing {'image': <bytes>, 'category': <str>} batches."""
    import ray
    if fmt == "lance":
        return ray.data.read_lance(lance_path, columns=["image", "category"])
    if fmt == "delta_inline":
        # Anti-pattern branch: image bytes stored INLINE in a binary column. No per-image
        # GET, but the SQL Warehouse must ship every image byte, and Parquet row-group
        # collapse (see parent README) makes the shuffled read scan far more than it needs.
        return ray.data.read_databricks_tables(
            warehouse_id=warehouse_id, catalog=catalog, schema=schema,
            query=f"SELECT image, category FROM {inline_delta_table}",
        )
    ds = ray.data.read_databricks_tables(
        warehouse_id=warehouse_id, catalog=catalog, schema=schema,
        query=f"SELECT image_path, category FROM {delta_table}",
    )
    return ds.map_batches(read_image_from_path, batch_size=io_batch_size)   # the per-image GET hop

## Section 1 — Training throughput

ResNet-50 (ImageNet-pretrained backbone, fresh head) trained with `TorchTrainer` + DDP across the A10 workers, streaming-shuffled read per epoch. Two runs per format: **dummy** (compute skipped → data-loading ceiling) and **real** (full step → I/O-vs-compute attribution).

### Shuffle: streaming, not global

| Strategy | What it does | Blocking? | Memory |
|----------|-------------|-----------|--------|
| **Global** (`.random_shuffle()`) | All-to-all reshuffle of the whole dataset before batching | Yes — GPUs idle until the full dataset is read + reshuffled; reads sequentially (a full scan) | Materializes decoded dataset in object store (spills at 1M/10M) |
| **Block-order + local buffer** *(used here)* | `randomize_block_order()` reorders which blocks read first; `local_shuffle_buffer` samples rows from a sliding window as they stream | No — pipelined, GPU stays fed | O(buffer_size) |

This benchmark uses the **streaming** strategy deliberately. A global shuffle reads sequentially and does the randomization afterward in the object store — which is exactly the pattern where the two formats look *most alike*, masking the difference the benchmark exists to show. `randomize_block_order()` varies the read order at the storage layer, exercising Lance's O(1) fragment-level random access vs Delta's row-group scans, while the local buffer supplies enough per-row entropy for SGD to converge. Non-blocking, and it sharpens the format signal rather than hiding it behind a full scan.

In [0]:
def train_fn_per_worker(config):
    import time
    import numpy as np
    import torch
    import torch.nn as nn
    import ray.train, ray.train.torch
    from torchvision.models import resnet50, ResNet50_Weights
    from mlflow.tracking import MlflowClient

    device   = ray.train.torch.get_device()
    rank     = ray.train.get_context().get_world_rank()
    dummy    = config["dummy"]
    n_epochs = config["num_epochs"]
    bs       = config["batch_size"]

    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    model.fc = nn.Linear(model.fc.in_features, config["num_classes"])
    model = ray.train.torch.prepare_model(model.to(device))
    opt   = torch.optim.SGD(model.parameters(), lr=1e-3, momentum=0.9)
    lossf = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler("cuda")

    shard  = ray.train.get_dataset_shard("train")
    client = MlflowClient() if rank == 0 else None

    for epoch in range(n_epochs):
        model.train()
        t_epoch = time.time()
        n, ttfb, batch_ms, wait_ms_list = 0, None, [], []
        t_wait_start = time.time()
        # Streaming per-worker shuffle + prefetch: samples rows from a sliding buffer as
        # the shard streams in (no blocking all-to-all barrier), and prefetch_batches
        # overlaps host->device copy with the current GPU step — keeps the GPU fed.
        for batch in shard.iter_torch_batches(
            batch_size=bs, dtypes=torch.float32, device=device,
            local_shuffle_buffer_size=config["local_shuffle_buffer"],
            prefetch_batches=config["prefetch_batches"],
        ):
            # Inter-batch wait: time spent blocking on data pipeline.
            wait_ms_list.append((time.time() - t_wait_start) * 1000)
            t_b = time.time()
            imgs = batch["image"]
            labels = batch["label"].long()
            if not dummy:
                with torch.amp.autocast("cuda"):
                    loss = lossf(model(imgs), labels)
                opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                _ = imgs.mean()
            if ttfb is None:
                ttfb = time.time() - t_epoch
            batch_ms.append((time.time() - t_b) * 1000)
            n += imgs.shape[0]
            t_wait_start = time.time()  # start timing wait for next batch

        dt = time.time() - t_epoch
        p_batch = np.percentile(batch_ms, [50, 95, 99])
        p_wait  = np.percentile(wait_ms_list, [50, 95, 99])
        metrics = {
            "samples_per_sec": n / dt, "epoch_wall_s": dt, "time_to_first_batch_s": ttfb,
            "batch_ms_p50": float(p_batch[0]), "batch_ms_p95": float(p_batch[1]), "batch_ms_p99": float(p_batch[2]),
            "wait_ms_p50":  float(p_wait[0]),  "wait_ms_p95":  float(p_wait[1]),  "wait_ms_p99":  float(p_wait[2]),
        }
        if client is not None:
            for k, v in metrics.items():
                client.log_metric(config["mlflow_run_id"], k, v, step=epoch)

        # Attach a checkpoint on the final epoch so result.metrics is populated.
        # Without a checkpoint, Ray Train 2.56+ leaves result.metrics empty.
        if epoch == n_epochs - 1:
            import tempfile
            with tempfile.TemporaryDirectory() as tmp_dir:
                ray.train.report(metrics, checkpoint=ray.train.Checkpoint.from_directory(tmp_dir))
        else:
            ray.train.report(metrics)

In [0]:
import mlflow
import pandas as pd
from ray.train.torch import TorchTrainer
from ray.train import ScalingConfig, RunConfig

mlflow.set_experiment(mlflow_exp)
training_results = {}

for fmt in FORMATS:
    for dummy in [True, False]:
        mode = "dummy" if dummy else "real"

        # randomize_block_order(): cheap per-epoch read-order entropy (varies storage-layer
        # access, exercising Lance O(1) random reads vs Delta row-group scans) with no
        # blocking all-to-all shuffle. Per-row entropy comes from local_shuffle_buffer.
        ds = read_format(fmt, warehouse_id, catalog, schema, delta_table, lance_path,
                         inline_delta_table=inline_delta_table).map_batches(
            decode_resize, fn_kwargs={"img_size": IMG_SIZE, "cat_to_idx": CAT_TO_IDX},
            batch_size=BATCH_SIZE,
        ).randomize_block_order()

        with mlflow.start_run(run_name=f"{fmt}_{mode}_{size}") as run:
            mlflow.log_params({"format": fmt, "mode": mode, "dataset_size": size,
                               "model": "resnet50", "batch_size": BATCH_SIZE,
                               "num_epochs": NUM_EPOCHS, "num_gpu_workers": NUM_GPU_WORKERS,
                               "prefetch_batches": PREFETCH_BATCHES,
                               "local_shuffle_buffer": LOCAL_SHUFFLE_BUFFER})
            trainer = TorchTrainer(
                train_loop_per_worker=train_fn_per_worker,
                train_loop_config={"dummy": dummy, "num_epochs": NUM_EPOCHS,
                                   "batch_size": BATCH_SIZE, "num_classes": len(CATEGORIES),
                                   "prefetch_batches": PREFETCH_BATCHES,
                                   "local_shuffle_buffer": LOCAL_SHUFFLE_BUFFER,
                                   "mlflow_run_id": run.info.run_id},
                scaling_config=ScalingConfig(num_workers=NUM_GPU_WORKERS,
                                              use_gpu=True,
                                              resources_per_worker={"CPU": 2, "GPU": 1}),
                datasets={"train": ds},
                # Shared FUSE path accessible from all nodes. The checkpoint on the
                # final epoch (cell 13) ensures result.metrics is populated in Ray 2.56+.
                run_config=RunConfig(storage_path=ray_tmp_path),
            )
            result = trainer.fit()
            if result.error:
                print(f"  ⚠ {fmt} {mode} FAILED: {result.error}")
            metrics = result.metrics or {}
            training_results[(fmt, mode)] = metrics
            if metrics:
                mlflow.log_metrics({k: v for k, v in metrics.items()
                                    if isinstance(v, (int, float))})
        print(f"  {fmt:8s} {mode:6s}: {metrics.get('samples_per_sec', 0):>10,.1f} samples/s")

# Clean up Ray Train artifacts from the shared volume.
import shutil
for entry in os.listdir(ray_tmp_path):
    if entry.startswith("ray_train_run"):
        shutil.rmtree(os.path.join(ray_tmp_path, entry), ignore_errors=True)

## Section 2 — Compare

Expected divergence (per the README): the per-image GET hop, projected reads, and shuffled random-access training throughput — not raw sequential scan. If the *dummy* gap is large but the *real* gap shrinks, GPU compute is masking the format difference — the tell to scale the model down or the data tier up.

In [0]:
rows = []
for fmt in FORMATS:
    rows.append({
        "format": fmt,
        # Dummy run = data-loading ceiling: streams + decodes but skips the GPU step.
        "load_sps":         round(training_results[(fmt, "dummy")].get("samples_per_sec", 0), 1),
        "train_real_sps":   round(training_results[(fmt, "real")].get("samples_per_sec", 0), 1),
        "real_ttfb_s":      round(training_results[(fmt, "real")].get("time_to_first_batch_s", 0), 2),
        "real_p99_ms":      round(training_results[(fmt, "real")].get("batch_ms_p99", 0), 1),
        "wait_ms_p50":      round(training_results[(fmt, "real")].get("wait_ms_p50", 0), 1),
        "wait_ms_p95":      round(training_results[(fmt, "real")].get("wait_ms_p95", 0), 1),
        })
summary = pd.DataFrame(rows)
display(summary)

l = summary[summary.format == "lance"].iloc[0]
d = summary[summary.format == "delta"].iloc[0]
if d.train_real_sps > 0:
    print(f"\nLance vs Delta (path-ref) @ {size}:")
    print(f"  data loading : {l.load_sps / max(1, d.load_sps):.2f}x")
    print(f"  train (real) : {l.train_real_sps / d.train_real_sps:.2f}x")

# Optional inline-Delta anti-pattern comparison.
if "delta_inline" in FORMATS:
    di = summary[summary.format == "delta_inline"].iloc[0]
    print(f"\nInline Delta (anti-pattern) @ {size}:")
    print(f"  Lance vs inline data loading : {l.load_sps / max(1, di.load_sps):.2f}x")
    if di.train_real_sps > 0:
        print(f"  Lance vs inline train (real) : {l.train_real_sps / di.train_real_sps:.2f}x")

In [0]:
# ── Persist training metrics to the shared artifacts volume (matches 01a / 01b) ──
import json, os

artifacts_dir = f"{base_vol}/artifacts"
os.makedirs(artifacts_dir, exist_ok=True)

training_metrics = {
    "size": size,
    "num_gpu_workers": NUM_GPU_WORKERS,
    "num_epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "prefetch_batches": PREFETCH_BATCHES,
    "local_shuffle_buffer": LOCAL_SHUFFLE_BUFFER,
    "formats": {},
}
for (fmt, mode), metrics in training_results.items():
    training_metrics["formats"].setdefault(fmt, {})[mode] = {
        k: round(v, 4) if isinstance(v, float) else v
        for k, v in metrics.items()
    }

out_path = f"{artifacts_dir}/training_{size}.json"
with open(out_path, "w") as f:
    json.dump(training_metrics, f, indent=2)
print(f"Wrote {out_path}")

## Deferred training metrics

Need node/GPU-level instrumentation rather than in-loop timing:

- **GPU utilization %** — `nvidia-smi` / DCGM sampled during the run (the direct data-starvation signal; samples/sec is the in-loop proxy used here).
- **Object-store spill events / disk IOPS** — Ray dashboard during shuffled reads.
- **Actor-pool utilization** and **CPU utilization on preprocess actors** — Ray dashboard timelines. For Delta, this includes the per-image GET actors, which is where its extra hop should show as IO-wait.